# M28 · Ads marketplace optimization

AFP-AI · Domain 6 · Optimization

This notebook simulates a tiny auction and pacing loop. Candidate value is $v=\widehat{pCTR} \times bid \times m$, where $m$ is a pacing multiplier.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(28)

## One auction first

We rank three ads by calibrated pCTR times bid. Then we apply a pacing multiplier to one advertiser and watch the winner change.

In [ ]:
ads = pd.DataFrame({
    "ad": ["A", "B", "C"],
    "pctr": [0.020, 0.030, 0.012],
    "bid": [8.0, 5.0, 11.0],
    "multiplier": [1.0, 1.0, 1.0]
})

ads["value"] = ads["pctr"] * ads["bid"] * ads["multiplier"]
ads = ads.sort_values("value", ascending=False).reset_index(drop=True)

print(ads)

## Second-price estimate

If the highest value wins, a normalized second-price CPC is approximately the runner-up value divided by the winner's pCTR.

In [ ]:
winner = ads.iloc[0]
runner_up = ads.iloc[1]
cpc = runner_up["value"] / winner["pctr"]
expected_spend = winner["pctr"] * cpc

print("winner:", winner["ad"])
print("estimated CPC:", round(cpc, 2))
print("expected spend per impression:", round(expected_spend, 3))

assert winner["ad"] == "A"
assert abs(cpc - 7.5) < 1e-9

## Apply pacing

Now A is ahead of budget, so its multiplier drops to $0.7$. The pCTR and bid stay the same; only the control signal changes.

In [ ]:
paced_ads = pd.DataFrame({
    "ad": ["A", "B", "C"],
    "pctr": [0.020, 0.030, 0.012],
    "bid": [8.0, 5.0, 11.0],
    "multiplier": [0.7, 1.0, 1.0]
})

paced_ads["value"] = paced_ads["pctr"] * paced_ads["bid"] * paced_ads["multiplier"]
paced_ads = paced_ads.sort_values("value", ascending=False).reset_index(drop=True)
paced_winner = paced_ads.iloc[0]
paced_runner_up = paced_ads.iloc[1]
paced_cpc = paced_runner_up["value"] / paced_winner["pctr"]

print(paced_ads)
print("paced winner:", paced_winner["ad"])
print("paced CPC:", round(paced_cpc, 2))

assert paced_winner["ad"] == "B"
assert abs(paced_cpc - 4.4) < 1e-9

## Simulate a day of auctions

We create 240 small auction opportunities. Advertiser A has a budget target; every 20 auctions, a feedback controller updates A's multiplier from spend error.

In [ ]:
n = 240
base_pctr = np.array([0.020, 0.026, 0.014])
bids = np.array([8.0, 5.5, 10.0])
multipliers = np.array([1.0, 1.0, 1.0])
budget_a = 18.0
gain = 0.8
spend_a = 0.0
spend_history = []
target_history = []
multiplier_history = []
wins = []

for t in range(n):
    noise = rng.normal(0.0, 0.003, size=3)
    pctr = np.clip(base_pctr + noise, 0.002, 0.08)
    values = pctr * bids * multipliers
    order = np.argsort(values)[::-1]
    winner_index = int(order[0])
    runner_index = int(order[1])
    price = values[runner_index] / pctr[winner_index]
    spend = pctr[winner_index] * price
    if winner_index == 0:
        spend_a = spend_a + spend
    if (t + 1) % 20 == 0:
        target = budget_a * (t + 1) / n
        error = (target - spend_a) / max(target, 1e-9)
        multipliers[0] = np.clip(multipliers[0] * (1.0 + gain * error), 0.2, 2.5)
    target = budget_a * (t + 1) / n
    spend_history.append(spend_a)
    target_history.append(target)
    multiplier_history.append(multipliers[0])
    wins.append(winner_index)

print("final A spend:", round(spend_a, 2))
print("budget target:", budget_a)
print("final A multiplier:", round(multipliers[0], 3))

assert spend_a <= budget_a * 1.35
assert spend_a >= budget_a * 0.65

## Visualize spend tracking

The controller is deliberately simple, but it shows the production idea: spend error changes future auction rank through the multiplier.

In [ ]:
steps = np.arange(1, n + 1)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(steps, spend_history, label="A spend")
axes[0].plot(steps, target_history, label="target")
axes[0].set_title("pacing spend vs target")
axes[0].set_xlabel("auction")
axes[0].set_ylabel("expected spend")
axes[0].legend()
axes[1].plot(steps, multiplier_history, color="#f58518")
axes[1].set_title("A pacing multiplier")
axes[1].set_xlabel("auction")
plt.tight_layout()
plt.show()

## Add a guardrail view

A marketplace choice can pass value ranking but fail a product guardrail. Here we summarize who won and check whether A stayed close enough to budget.

In [ ]:
win_counts = pd.Series(wins).map({0: "A", 1: "B", 2: "C"}).value_counts().sort_index()
budget_error = (spend_a - budget_a) / budget_a

print(win_counts)
print("budget error:", round(budget_error, 3))

assert abs(budget_error) < 0.35

## Your turn

1. Increase A's bid from 8 to 10 and re-run the loop.
2. Lower the gain from 0.8 to 0.2 and compare tracking.
3. Add a quality multiplier $q$ and rank by $q \times pCTR \times bid \times m$.

In [ ]:
# Your turn:
